# Asia Pacific Local Governance Project Mapping Dashboard  
## Data Preparation Notebook for Power BI

**Notebook:** 01_prepare_undp_project_mapping_data.ipynb  
**Project Type:** Data preparation, cleaning, and dashboard-ready Excel export  
**Dashboard Tool:** Microsoft Power BI  
**Data Source:** Public UNDP Transparency Portal / IATI project data  

## 1. Project Introduction

This notebook prepares a sample Asia Pacific local governance project mapping dataset for an interactive Power BI dashboard.

The project is designed as a portfolio demonstration for transforming public development project data into a dashboard-ready format. It uses publicly available UNDP Transparency Portal / IATI project data from selected Asia Pacific countries and converts it into a structured Excel file suitable for Power BI visualization.

The final Power BI dashboard will support project exploration through country filters, project status analysis, thematic focus areas, SDG alignment, donor and budget views, geographic mapping, and project-level drill-through pages.

## 2. Objective

The objective of this notebook is to create a clean and structured Excel dataset that can be used to build an interactive Power BI dashboard for local governance and local development project mapping.

More specifically, this notebook will:

1. Load public UNDP project data for selected Asia Pacific countries.
2. Extract relevant project fields such as project title, country, status, dates, budget, sector, donor, and description.
3. Clean and standardize text, date, budget, and status fields.
4. Add dashboard-ready analytical columns such as SDG alignment, thematic focus, intervention type, and geographic coverage.
5. Add country coordinates for Power BI map visualization.
6. Export a clean Excel file for Power BI dashboard development.

## 3. Relevance to Local Governance Project Mapping

The dashboard being prepared from this dataset is aligned with the requirements of a local governance and local development project mapping dashboard.

The expected dashboard will include:

1. Geographic map visualization of projects across Asia Pacific countries.
2. Filter panels and slicers for country, project status, thematic focus, SDG alignment, donor, and budget.
3. Summary statistics such as total projects, countries covered, total budget, active projects, and donor count.
4. Charts showing projects by country, status, SDG, donor, thematic focus, and intervention type.
5. A project detail page for drill-through analysis.
6. A clean layout suitable for web-accessible reporting and stakeholder review.

## 4. Data Source

This notebook uses public UNDP Transparency Portal / IATI project data.

The source data is downloaded from public country-level XML files hosted by UNDP. These files contain project-level development information such as activity identifiers, titles, descriptions, dates, sectors, budgets, and participating organizations.

For this portfolio dashboard, selected Asia Pacific countries are used to create a realistic sample project mapping dataset. The dataset is intended for demonstration and learning purposes only.

## 5. Methodology Note

Some fields required for dashboard analysis may not appear in the source XML files in a clean dashboard-ready format. Therefore, this notebook creates selected analytical classifications using keyword-based logic.

The following fields are generated for dashboard demonstration:

1. SDG Alignment
2. Thematic Focus
3. Intervention Type
4. Geographic Coverage

These classifications are based on keywords found in project titles, descriptions, and sector information. They should be treated as analytical tags created for portfolio demonstration, not as official UNDP classifications.

## 6. Expected Output

The final output of this notebook will be a clean Excel file:

`asia_pacific_undp_project_mapping_sample.xlsx`

This Excel file will be used as the source dataset for the Power BI dashboard.

The final dataset will include the following dashboard-ready fields:

1. Project ID
2. Project Title
3. Country
4. Latitude
5. Longitude
6. Thematic Focus
7. SDG Alignment
8. Intervention Type
9. Donor / Funding Source
10. Budget USD
11. Project Status
12. Start Date
13. End Date
14. Start Year
15. End Year
16. Geographic Coverage
17. Sector Raw
18. Project Description
19. Source File

## 7. Data Preparation

The following section imports the required Python libraries, creates the project folder structure, downloads the public UNDP XML data files, and prepares the dataset for cleaning and transformation.

In [1]:
import pandas as pd
import numpy as np
import requests
import xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime
import re

In [4]:
# Project folders
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


In [5]:
country_sources = {
    "Thailand": "https://open.undp.org/download/iati_xml/Thailand_projects.xml",
    "Indonesia": "https://open.undp.org/download/iati_xml/Indonesia_projects.xml",
    "Cambodia": "https://open.undp.org/download/iati_xml/Cambodia_projects.xml",
    "Viet Nam": "https://open.undp.org/download/iati_xml/Viet_Nam_projects.xml",
    "Bangladesh": "https://open.undp.org/download/iati_xml/Bangladesh_projects.xml",
    "Pakistan": "https://open.undp.org/download/iati_xml/Pakistan_projects.xml",
    "Nepal": "https://open.undp.org/download/iati_xml/Nepal_projects.xml",
    "Sri Lanka": "https://open.undp.org/download/iati_xml/Sri_Lanka_projects.xml",
    "Philippines": "https://open.undp.org/download/iati_xml/Philippines_projects.xml",
    "Papua New Guinea": "https://open.undp.org/download/iati_xml/Papua_New_Guinea_projects.xml"
}

country_sources

{'Thailand': 'https://open.undp.org/download/iati_xml/Thailand_projects.xml',
 'Indonesia': 'https://open.undp.org/download/iati_xml/Indonesia_projects.xml',
 'Cambodia': 'https://open.undp.org/download/iati_xml/Cambodia_projects.xml',
 'Viet Nam': 'https://open.undp.org/download/iati_xml/Viet_Nam_projects.xml',
 'Bangladesh': 'https://open.undp.org/download/iati_xml/Bangladesh_projects.xml',
 'Pakistan': 'https://open.undp.org/download/iati_xml/Pakistan_projects.xml',
 'Nepal': 'https://open.undp.org/download/iati_xml/Nepal_projects.xml',
 'Sri Lanka': 'https://open.undp.org/download/iati_xml/Sri_Lanka_projects.xml',
 'Philippines': 'https://open.undp.org/download/iati_xml/Philippines_projects.xml',
 'Papua New Guinea': 'https://open.undp.org/download/iati_xml/Papua_New_Guinea_projects.xml'}

In [6]:
def download_xml_files(country_sources, raw_dir):
    downloaded_files = {}

    for country, url in country_sources.items():
        safe_country_name = country.replace(" ", "_")
        file_path = raw_dir / f"{safe_country_name}_projects.xml"

        print(f"Downloading: {country}")

        try:
            response = requests.get(url, timeout=60)
            response.raise_for_status()

            file_path.write_bytes(response.content)
            downloaded_files[country] = file_path

            print(f"Saved: {file_path.name}")

        except requests.exceptions.RequestException as error:
            print(f"Failed for {country}: {error}")

    return downloaded_files


downloaded_files = download_xml_files(country_sources, RAW_DIR)

print("\nTotal files downloaded:", len(downloaded_files))

Downloading: Thailand
Saved: Thailand_projects.xml
Downloading: Indonesia
Saved: Indonesia_projects.xml
Downloading: Cambodia
Saved: Cambodia_projects.xml
Downloading: Viet Nam
Saved: Viet_Nam_projects.xml
Downloading: Bangladesh
Saved: Bangladesh_projects.xml
Downloading: Pakistan
Saved: Pakistan_projects.xml
Downloading: Nepal
Saved: Nepal_projects.xml
Downloading: Sri Lanka
Saved: Sri_Lanka_projects.xml
Downloading: Philippines
Saved: Philippines_projects.xml
Downloading: Papua New Guinea
Saved: Papua_New_Guinea_projects.xml

Total files downloaded: 10


In [7]:
def get_text(element):
    """
    Safely extract text from an XML element.
    """
    if element is None:
        return None

    if element.text is None:
        return None

    return element.text.strip()


def get_first_narrative(parent, tag_name):
    """
    Extract first narrative text from elements like title, description, etc.
    IATI XML often stores text inside <narrative>.
    """
    if parent is None:
        return None

    element = parent.find(tag_name)
    if element is None:
        return None

    narrative = element.find("narrative")
    return get_text(narrative)


def clean_text(value):
    """
    Clean text fields for dashboard use.
    """
    if pd.isna(value):
        return None

    value = str(value)
    value = re.sub(r"\s+", " ", value)
    value = value.replace("\n", " ").replace("\r", " ")
    value = value.strip()

    return value if value else None


def parse_date(value):
    """
    Convert date strings to pandas datetime.
    """
    if value is None:
        return pd.NaT

    try:
        return pd.to_datetime(value, errors="coerce")
    except Exception:
        return pd.NaT

In [8]:
def parse_undp_iati_xml(file_path, country_name):
    """
    Parse UNDP IATI XML and extract dashboard-ready project fields.
    """
    records = []

    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
    except ET.ParseError as error:
        print(f"XML parse error in {file_path.name}: {error}")
        return pd.DataFrame()

    activities = root.findall(".//iati-activity")

    for activity in activities:
        project_id = get_text(activity.find("iati-identifier"))

        title = get_first_narrative(activity, "title")

        description = None
        descriptions = activity.findall("description")
        if descriptions:
            description = get_text(descriptions[0].find("narrative"))

        activity_status = activity.find("activity-status")
        status_code = activity_status.attrib.get("code") if activity_status is not None else None

        start_date = None
        end_date = None

        for date_element in activity.findall("activity-date"):
            date_type = date_element.attrib.get("type")
            iso_date = date_element.attrib.get("iso-date")

            if date_type in ["1", "2"] and start_date is None:
                start_date = iso_date

            if date_type in ["3", "4"]:
                end_date = iso_date

        sectors = activity.findall("sector")
        sector_names = []

        for sector in sectors:
            narrative = sector.find("narrative")
            sector_text = get_text(narrative)

            if sector_text:
                sector_names.append(sector_text)
            else:
                sector_code = sector.attrib.get("code")
                if sector_code:
                    sector_names.append(f"Sector code {sector_code}")

        participating_orgs = activity.findall("participating-org")
        donor_names = []

        for org in participating_orgs:
            role = org.attrib.get("role")
            org_name = get_text(org.find("narrative"))

            if org_name and role in ["1", "3", "4", None]:
                donor_names.append(org_name)

        total_budget = 0

        for budget in activity.findall("budget"):
            value_element = budget.find("value")
            value_text = get_text(value_element)

            if value_text:
                try:
                    total_budget += float(value_text)
                except ValueError:
                    pass

        record = {
            "Project ID": project_id,
            "Project Title": title,
            "Country": country_name,
            "Project Status Code": status_code,
            "Start Date": start_date,
            "End Date": end_date,
            "Sector Raw": "; ".join(sorted(set(sector_names))) if sector_names else None,
            "Donor / Funding Source": "; ".join(sorted(set(donor_names))) if donor_names else None,
            "Budget USD": total_budget if total_budget > 0 else np.nan,
            "Project Description": description,
            "Source File": file_path.name
        }

        records.append(record)

    return pd.DataFrame(records)

In [9]:
all_country_data = []

for country, file_path in downloaded_files.items():
    print(f"Parsing: {country}")

    country_df = parse_undp_iati_xml(file_path, country)

    print(f"Rows extracted: {len(country_df)}")

    all_country_data.append(country_df)

projects_raw = pd.concat(all_country_data, ignore_index=True)

print("\nTotal raw projects:", len(projects_raw))
projects_raw.head()

Parsing: Thailand
Rows extracted: 61
Parsing: Indonesia
Rows extracted: 152
Parsing: Cambodia
Rows extracted: 87
Parsing: Viet Nam
Rows extracted: 151
Parsing: Bangladesh
Rows extracted: 120
Parsing: Pakistan
Rows extracted: 107
Parsing: Nepal
Rows extracted: 142
Parsing: Sri Lanka
Rows extracted: 143
Parsing: Philippines
Rows extracted: 163
Parsing: Papua New Guinea
Rows extracted: 109

Total raw projects: 1235


,Project ID,Project Title,Country,Project Status Code,Start Date,End Date,Sector Raw,Donor / Funding Source,Budget USD,Project Description,Source File
0,XM-DAC-41114-PROJECT-00047052,Development Advisory Services,Thailand,3,2007-04-18,2024-03-01,Sector code 12.7; Sector code 15110; Sector co...,GOVERNMENT OF POLAND; Securities & Exchange Co...,NaN,Development Advisory Services for project prep...,Thailand_projects.xml
1,XM-DAC-41114-OUTPUT-00056302,Development Advisory Services,Thailand,3,2007-04-18,2024-03-01,Sector code 12.7; Sector code 15110; Sector co...,GOVERNMENT OF POLAND; Securities & Exchange Co...,1353715.0,Development Advisory Services (DAS) are intend...,Thailand_projects.xml
2,XM-DAC-41114-PROJECT-00064108,Development Effectiveness,Thailand,2,2012-01-01,2030-12-31,Sector code 91010,UNITED NATIONS DEVELOPMENT PROGRAMME; United N...,NaN,Development Effectiveness is a management proj...,Thailand_projects.xml
3,XM-DAC-41114-OUTPUT-00109016,Direct Project Cost,Thailand,2,2018-01-01,2030-12-31,Sector code 91010,UNITED NATIONS DEVELOPMENT PROGRAMME; United N...,393481.0,Direct Project Cost for staff who contributes ...,Thailand_projects.xml
4,XM-DAC-41114-PROJECT-00086286,Strengthening Enforcement Networks Forensic Te...,Thailand,3,2018-05-01,2025-06-01,Sector code 12.8; Sector code 15.2; Sector cod...,DNP- Min. of Natural Resources; Global Environ...,NaN,"Strengthening Enforcement Networks, Forensic T...",Thailand_projects.xml


In [10]:
projects = projects_raw.copy()

text_columns = [
    "Project ID",
    "Project Title",
    "Country",
    "Sector Raw",
    "Donor / Funding Source",
    "Project Description",
    "Source File"
]

for col in text_columns:
    projects[col] = projects[col].apply(clean_text)

projects["Start Date"] = projects["Start Date"].apply(parse_date)
projects["End Date"] = projects["End Date"].apply(parse_date)

projects["Start Year"] = projects["Start Date"].dt.year
projects["End Year"] = projects["End Date"].dt.year

projects["Budget USD"] = pd.to_numeric(projects["Budget USD"], errors="coerce")
projects["Budget USD"] = projects["Budget USD"].fillna(0)

projects = projects.drop_duplicates(subset=["Project ID", "Country"], keep="first")

projects.head()

,Project ID,Project Title,Country,Project Status Code,Start Date,End Date,Sector Raw,Donor / Funding Source,Budget USD,Project Description,Source File,Start Year,End Year
0,XM-DAC-41114-PROJECT-00047052,Development Advisory Services,Thailand,3,2007-04-18,2024-03-01,Sector code 12.7; Sector code 15110; Sector co...,GOVERNMENT OF POLAND; Securities & Exchange Co...,0.0,Development Advisory Services for project prep...,Thailand_projects.xml,2007,2024
1,XM-DAC-41114-OUTPUT-00056302,Development Advisory Services,Thailand,3,2007-04-18,2024-03-01,Sector code 12.7; Sector code 15110; Sector co...,GOVERNMENT OF POLAND; Securities & Exchange Co...,1353715.0,Development Advisory Services (DAS) are intend...,Thailand_projects.xml,2007,2024
2,XM-DAC-41114-PROJECT-00064108,Development Effectiveness,Thailand,2,2012-01-01,2030-12-31,Sector code 91010,UNITED NATIONS DEVELOPMENT PROGRAMME; United N...,0.0,Development Effectiveness is a management proj...,Thailand_projects.xml,2012,2030
3,XM-DAC-41114-OUTPUT-00109016,Direct Project Cost,Thailand,2,2018-01-01,2030-12-31,Sector code 91010,UNITED NATIONS DEVELOPMENT PROGRAMME; United N...,393481.0,Direct Project Cost for staff who contributes ...,Thailand_projects.xml,2018,2030
4,XM-DAC-41114-PROJECT-00086286,Strengthening Enforcement Networks Forensic Te...,Thailand,3,2018-05-01,2025-06-01,Sector code 12.8; Sector code 15.2; Sector cod...,DNP- Min. of Natural Resources; Global Environ...,0.0,"Strengthening Enforcement Networks, Forensic T...",Thailand_projects.xml,2018,2025


In [33]:
status_map = {
    "1": "Pipeline / Identification",
    "2": "Implementation",
    "3": "Finalisation",
    "4": "Closed",
    "5": "Cancelled",
    "6": "Suspended"
}

projects["Project Status"] = projects["Project Status Code"].map(status_map)
projects["Project Status"] = projects["Project Status"].fillna("Not Specified")

projects[["Project ID", "Project Title", "Country", "Project Status"]].head()

,Project ID,Project Title,Country,Project Status
0,XM-DAC-41114-PROJECT-00047052,Development Advisory Services,Thailand,Finalisation
1,XM-DAC-41114-OUTPUT-00056302,Development Advisory Services,Thailand,Finalisation
2,XM-DAC-41114-PROJECT-00064108,Development Effectiveness,Thailand,Implementation
3,XM-DAC-41114-OUTPUT-00109016,Direct Project Cost,Thailand,Implementation
4,XM-DAC-41114-PROJECT-00086286,Strengthening Enforcement Networks Forensic Te...,Thailand,Finalisation


In [34]:
def create_dashboard_status(row):
    """
    Create a dashboard-friendly project status field using start and end years.
    The original Project Status is kept separately.
    """
    start_year = row.get("Start Year")
    end_year = row.get("End Year")

    current_year = datetime.today().year

    if pd.notna(end_year):
        if end_year >= current_year + 1:
            return "Active"
        elif end_year == current_year:
            return "Closing"
        elif end_year < current_year:
            return "Completed"

    if pd.notna(start_year):
        if start_year >= current_year:
            return "Pipeline"
        elif start_year < current_year:
            return "Active"

    return "Not Specified"


projects["Dashboard Status"] = projects.apply(create_dashboard_status, axis=1)

projects["Dashboard Status"].value_counts()

Dashboard Status
Closing      425
Completed    419
Active       391
Name: count, dtype: int64

In [35]:
country_coordinates = {
    "Thailand": {"Latitude": 15.8700, "Longitude": 100.9925},
    "Indonesia": {"Latitude": -0.7893, "Longitude": 113.9213},
    "Cambodia": {"Latitude": 12.5657, "Longitude": 104.9910},
    "Viet Nam": {"Latitude": 14.0583, "Longitude": 108.2772},
    "Bangladesh": {"Latitude": 23.6850, "Longitude": 90.3563},
    "Pakistan": {"Latitude": 30.3753, "Longitude": 69.3451},
    "Nepal": {"Latitude": 28.3949, "Longitude": 84.1240},
    "Sri Lanka": {"Latitude": 7.8731, "Longitude": 80.7718},
    "Philippines": {"Latitude": 12.8797, "Longitude": 121.7740},
    "Papua New Guinea": {"Latitude": -6.3150, "Longitude": 143.9555}
}

projects["Latitude"] = projects["Country"].map(lambda x: country_coordinates.get(x, {}).get("Latitude"))
projects["Longitude"] = projects["Country"].map(lambda x: country_coordinates.get(x, {}).get("Longitude"))

projects[["Country", "Latitude", "Longitude"]].drop_duplicates()

,Country,Latitude,Longitude
0,Thailand,15.8700,100.9925
61,Indonesia,-0.7893,113.9213
213,Cambodia,12.5657,104.9910
300,Viet Nam,14.0583,108.2772
451,Bangladesh,23.6850,90.3563
571,Pakistan,30.3753,69.3451
678,Nepal,28.3949,84.1240
820,Sri Lanka,7.8731,80.7718
963,Philippines,12.8797,121.7740
1126,Papua New Guinea,-6.3150,143.9555


In [36]:
def classify_sdg(row):
    text = " ".join([
        str(row.get("Project Title", "")),
        str(row.get("Sector Raw", "")),
        str(row.get("Project Description", ""))
    ]).lower()

    sdgs = []

    if any(word in text for word in ["governance", "justice", "peace", "institution", "rule of law", "anti-corruption", "parliament", "election"]):
        sdgs.append("SDG 16")

    if any(word in text for word in ["partnership", "coordination", "donor", "finance", "cooperation"]):
        sdgs.append("SDG 17")

    if any(word in text for word in ["city", "urban", "municipal", "local government", "local governance", "community", "settlement"]):
        sdgs.append("SDG 11")

    if any(word in text for word in ["climate", "resilience", "disaster", "flood", "environment", "adaptation"]):
        sdgs.append("SDG 13")

    if any(word in text for word in ["poverty", "livelihood", "income", "employment", "jobs"]):
        sdgs.append("SDG 1 / SDG 8")

    if any(word in text for word in ["gender", "women", "girls"]):
        sdgs.append("SDG 5")

    if not sdgs:
        sdgs.append("Other SDG")

    return "; ".join(sorted(set(sdgs)))


projects["SDG Alignment"] = projects.apply(classify_sdg, axis=1)

projects[["Project Title", "SDG Alignment"]].head(10)

,Project Title,SDG Alignment
0,Development Advisory Services,SDG 17
1,Development Advisory Services,Other SDG
2,Development Effectiveness,Other SDG
3,Direct Project Cost,Other SDG
4,Strengthening Enforcement Networks Forensic Te...,Other SDG
5,Combating Illegal Wildlife Tra,SDG 11
6,Enhancing Climate Resilience in Thailand throu...,SDG 13
7,Enhancing Climate Resilience i,SDG 13
8,Increasing resilience to climate change impacts,SDG 13
9,6032 Thailand National Adaptation,SDG 13


In [39]:
def classify_thematic_focus(row):
    text = " ".join([
        str(row.get("Project Title", "")),
        str(row.get("Sector Raw", "")),
        str(row.get("Project Description", ""))
    ]).lower()

    if any(word in text for word in ["governance", "local government", "local governance", "institution", "public administration", "rule of law"]):
        return "Local Governance and Institutions"

    if any(word in text for word in ["climate", "resilience", "disaster", "flood", "environment", "adaptation"]):
        return "Climate Resilience and Environment"

    if any(word in text for word in ["poverty", "livelihood", "employment", "jobs", "enterprise", "economic"]):
        return "Inclusive Economic Development"

    if any(word in text for word in ["gender", "women", "girls", "youth"]):
        return "Gender Equality and Social Inclusion"

    if any(word in text for word in ["digital", "innovation", "data", "technology", "e-governance"]):
        return "Digital Governance and Innovation"

    if any(word in text for word in ["health", "education", "social protection", "community"]):
        return "Social Services and Community Development"

    return "Other Development Area"


projects["Thematic Focus"] = projects.apply(classify_thematic_focus, axis=1)

projects["Thematic Focus"].value_counts()

Thematic Focus
Climate Resilience and Environment           370
Local Governance and Institutions            285
Other Development Area                       229
Digital Governance and Innovation            199
Inclusive Economic Development                93
Gender Equality and Social Inclusion          40
Social Services and Community Development     19
Name: count, dtype: int64

In [38]:
def classify_intervention_type(row):
    text = " ".join([
        str(row.get("Project Title", "")),
        str(row.get("Sector Raw", "")),
        str(row.get("Project Description", ""))
    ]).lower()

    if any(word in text for word in ["capacity", "training", "workshop", "skills"]):
        return "Capacity Building"

    if any(word in text for word in ["policy", "strategy", "framework", "reform", "planning"]):
        return "Policy and Institutional Support"

    if any(word in text for word in ["digital", "platform", "data", "system", "technology", "innovation"]):
        return "Digital Solution / Data System"

    if any(word in text for word in ["community", "local", "municipal", "participation", "citizen"]):
        return "Community and Local Engagement"

    if any(word in text for word in ["assessment", "study", "research", "analysis", "mapping"]):
        return "Research, Mapping and Assessment"

    return "Programme Implementation Support"


projects["Intervention Type"] = projects.apply(classify_intervention_type, axis=1)

projects["Intervention Type"].value_counts()

Intervention Type
Programme Implementation Support    372
Digital Solution / Data System      326
Policy and Institutional Support    233
Capacity Building                   212
Community and Local Engagement       85
Research, Mapping and Assessment      7
Name: count, dtype: int64

In [40]:
def classify_geographic_coverage(row):
    text = " ".join([
        str(row.get("Project Title", "")),
        str(row.get("Project Description", ""))
    ]).lower()

    if any(word in text for word in ["municipal", "city", "urban", "district", "province", "provincial", "local"]):
        return "Sub-national / Local"

    if any(word in text for word in ["regional", "asia", "pacific", "cross-border"]):
        return "Regional"

    return "National"


projects["Geographic Coverage"] = projects.apply(classify_geographic_coverage, axis=1)

projects["Geographic Coverage"].value_counts()

Geographic Coverage
National                800
Sub-national / Local    401
Regional                 34
Name: count, dtype: int64

In [42]:
final_columns = [
    "Project ID",
    "Project Title",
    "Country",
    "Latitude",
    "Longitude",
    "Thematic Focus",
    "SDG Alignment",
    "Intervention Type",
    "Donor / Funding Source",
    "Budget USD",
    "Project Status",
    "Dashboard Status",
    "Start Date",
    "End Date",
    "Start Year",
    "End Year",
    "Geographic Coverage",
    "Sector Raw",
    "Project Description",
    "Source File"
]

dashboard_projects = projects[final_columns].copy()

dashboard_projects = dashboard_projects.sort_values(
    by=["Country", "Project Status", "Project Title"],
    ascending=[True, True, True]
).reset_index(drop=True)

dashboard_projects.head()

,Project ID,Project Title,Country,Latitude,Longitude,Thematic Focus,SDG Alignment,Intervention Type,Donor / Funding Source,Budget USD,Project Status,Dashboard Status,Start Date,End Date,Start Year,End Year,Geographic Coverage,Sector Raw,Project Description,Source File
0,XM-DAC-41114-OUTPUT-00091274,Activating Village Court II,Bangladesh,23.685,90.3563,Digital Governance and Innovation,Other SDG,Digital Solution / Data System,EUROPEAN COMMISSION; Local Government Division...,29063559.0,Finalisation,Completed,2016-01-01,2024-12-01,2016,2024,Sub-national / Local,Sector code 15131; Sector code 16.3; Structura...,Activating Village Court Bangladesh Phase II (...,Bangladesh_projects.xml
1,XM-DAC-41114-PROJECT-00082279,Activating Village Court Phase II,Bangladesh,23.685,90.3563,Other Development Area,Other SDG,Community and Local Engagement,EUROPEAN COMMISSION; Local Government Division...,0.0,Finalisation,Completed,2016-01-01,2024-12-01,2016,2024,Sub-national / Local,Sector code 15131; Sector code 16.3,Activating Village Courts in Bangladesh Phase ...,Bangladesh_projects.xml
2,XM-DAC-41114-OUTPUT-00115133,Activating Village Court inCHT,Bangladesh,23.685,90.3563,Digital Governance and Innovation,Other SDG,Digital Solution / Data System,EUROPEAN COMMISSION; Local Government Division,6322288.0,Finalisation,Completed,2019-01-01,2024-10-01,2019,2024,Sub-national / Local,Sector code 15131; Sector code 16.3; Structura...,Activating Village Courts in Bangladesh Phase ...,Bangladesh_projects.xml
3,XM-DAC-41114-OUTPUT-00113358,Community Cohesion in CXB,Bangladesh,23.685,90.3563,Climate Resilience and Environment,SDG 11; SDG 13,Digital Solution / Data System,Dept of Forgn Afrs Trade & Dev; TAR-Canada Dep...,6209861.0,Finalisation,Completed,2018-12-10,2024-03-01,2018,2024,National,Sector code 16.1; Sector code 16.6; Sector cod...,Support to strengthen economic resilience of t...,Bangladesh_projects.xml
4,XM-DAC-41114-OUTPUT-00112092,DRRF-Disaster Resp& Recov.Faci,Bangladesh,23.685,90.3563,Climate Resilience and Environment,SDG 11; SDG 13,Community and Local Engagement,ClimateWorks Foundation; GOVERNMENT OF GERMANY...,6288374.0,Finalisation,Completed,2018-10-01,2024-11-01,2018,2024,National,Building resilience; Sector code 13.1; Sector ...,Disaster Response and Recovery Facility (DRRF)...,Bangladesh_projects.xml


In [43]:
print("Rows:", len(dashboard_projects))
print("Countries:", dashboard_projects["Country"].nunique())
print("Total Budget USD:", dashboard_projects["Budget USD"].sum())
print("\nMissing values:")
print(dashboard_projects.isna().sum())

print("\nProject status distribution:")
print(dashboard_projects["Project Status"].value_counts())

print("\nThematic focus distribution:")
print(dashboard_projects["Thematic Focus"].value_counts())

Rows: 1235
Countries: 10
Total Budget USD: 3018690220.0

Missing values:
Project ID                0
Project Title             0
Country                   0
Latitude                  0
Longitude                 0
Thematic Focus            0
SDG Alignment             0
Intervention Type         0
Donor / Funding Source    0
Budget USD                0
Project Status            0
Dashboard Status          0
Start Date                0
End Date                  0
Start Year                0
End Year                  0
Geographic Coverage       0
Sector Raw                0
Project Description       0
Source File               0
dtype: int64

Project status distribution:
Project Status
Implementation    882
Finalisation      353
Name: count, dtype: int64

Thematic focus distribution:
Thematic Focus
Climate Resilience and Environment           370
Local Governance and Institutions            285
Other Development Area                       229
Digital Governance and Innovation            19

In [44]:
# Create a balanced sample for portfolio dashboard
# Priority is to include variation in project status where available

dashboard_sample = (
    dashboard_projects
    .sort_values(by=["Country", "Project Status", "Project Title"])
    .groupby("Country", group_keys=False)
    .apply(lambda x: x.head(10))
    .reset_index(drop=True)
)

print("Sample rows:", len(dashboard_sample))
print("Countries:", dashboard_sample["Country"].nunique())

print("\nProject status distribution:")
print(dashboard_sample["Project Status"].value_counts())

Sample rows: 100
Countries: 10

Project status distribution:
Project Status
Finalisation    100
Name: count, dtype: int64


C:\Users\LP\AppData\Local\Temp\ipykernel_28964\3417347514.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.head(10))


In [45]:
output_file = PROCESSED_DIR / "asia_pacific_undp_project_mapping_sample.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    dashboard_sample.to_excel(writer, sheet_name="Projects", index=False)

print("Excel file created successfully:")


Excel file created successfully:


In [47]:
metadata = pd.DataFrame({
    "Item": [
        "Project",
        "Data Source",
        "Purpose",
        "Countries Included",
        "Rows Exported",
        "Date Prepared",
        "Important Note"
    ],
    "Description": [
        "Asia Pacific Local Governance Project Mapping Dashboard",
        "Public UNDP Transparency Portal IATI XML project data",
        "Portfolio demonstration for Power BI dashboard development",
        ", ".join(sorted(dashboard_sample["Country"].unique())),
        len(dashboard_sample),
        datetime.today().strftime("%Y-%m-%d"),
        "SDG Alignment, Thematic Focus, Intervention Type, and Geographic Coverage are analytical classifications created from project text for dashboard demonstration."
    ]
})

metadata_file = PROCESSED_DIR / "dataset_metadata.xlsx"

with pd.ExcelWriter(metadata_file, engine="openpyxl") as writer:
    metadata.to_excel(writer, sheet_name="Metadata", index=False)

print("Metadata file created:")


Metadata file created:


In [48]:
dashboard_sample.sample(10, random_state=42)

,Project ID,Project Title,Country,Latitude,Longitude,Thematic Focus,SDG Alignment,Intervention Type,Donor / Funding Source,Budget USD,Project Status,Dashboard Status,Start Date,End Date,Start Year,End Year,Geographic Coverage,Sector Raw,Project Description,Source File
83,XM-DAC-41114-OUTPUT-00056302,Development Advisory Services,Thailand,15.8700,100.9925,Digital Governance and Innovation,Other SDG,Policy and Institutional Support,GOVERNMENT OF POLAND; Securities & Exchange Co...,1353715.0,Finalisation,Completed,2007-04-18,2024-03-01,2007,2024,National,Sector code 12.7; Sector code 15110; Sector co...,Development Advisory Services (DAS) are intend...,Thailand_projects.xml
53,XM-DAC-41114-PROJECT-00095522,Communications and Monitoring Evaluation,Papua New Guinea,-6.3150,143.9555,Other Development Area,Other SDG,Programme Implementation Support,EUROPEAN COMMISSION; UNITED NATIONS DEVELOPMEN...,0.0,Finalisation,Completed,2016-01-01,2025-06-01,2016,2025,National,Sector code 99810,Communications and Monitoring & Evaluation Sup...,Papua_New_Guinea_projects.xml
70,XM-DAC-41114-OUTPUT-00096013,Biomass Phase 2,Sri Lanka,7.8731,80.7718,Digital Governance and Innovation,Other SDG,Digital Solution / Data System,GOVERNMENT OF SRI LANKA; PRIVATE SECTOR; UNITE...,907156.0,Finalisation,Completed,2018-08-01,2024-09-01,2018,2024,Sub-national / Local,Sector code 23210; Sector code 7.1; Sector cod...,Biomass Energy 2022 will consolidate the resul...,Sri_Lanka_projects.xml
45,XM-DAC-41114-OUTPUT-00084531,Decentralization & Local Gover,Pakistan,30.3753,69.3451,Local Governance and Institutions,SDG 16,Digital Solution / Data System,Australian DFAT; DEPARTMENT FOR INTERNATIONAL ...,13547699.0,Finalisation,Completed,2012-11-01,2025-06-01,2012,2025,Sub-national / Local,Sector code 10.3; Sector code 15112; Sector co...,The project identifies strategic areas of inte...,Pakistan_projects.xml
44,XM-DAC-41114-OUTPUT-00116110,Climate Change Adaptation and Mitigation in Pa...,Pakistan,30.3753,69.3451,Local Governance and Institutions,SDG 13; SDG 16,Policy and Institutional Support,FOOD AND AGRICULTURE ORGANIZATION OF THE UNITE...,1672796.0,Finalisation,Completed,2019-01-01,2024-03-01,2019,2024,National,Sector code 1.5; Sector code 12.2; Sector code...,Institutional Support to Climate Change Adapta...,Pakistan_projects.xml
39,XM-DAC-41114-OUTPUT-00104559,Cooperative Market Development,Nepal,28.3949,84.1240,Digital Governance and Innovation,SDG 11,Capacity Building,GOVERNMENT OF NEPAL; GUY-GOVERNMENT OF GUYANA;...,7870496.0,Finalisation,Completed,2018-02-02,2024-11-01,2018,2024,Sub-national / Local,Sector code 1.2; Sector code 16050; Sector cod...,Capacity of Fruits and Vegetable production co...,Nepal_projects.xml
22,XM-DAC-41114-OUTPUT-00110412,ATSEA 2 Regional,Indonesia,-0.7893,113.9213,Climate Resilience and Environment,SDG 13,Digital Solution / Data System,Global Environment Fund Truste; KWARA STATE GO...,4738022.0,Finalisation,Completed,2019-05-01,2025-06-01,2019,2025,Regional,Building resilience; Sector code 14.2; Sector ...,The ATSEA-2 is designed to enhance sustainable...,Indonesia_projects.xml
80,XM-DAC-41114-OUTPUT-00093576,Combating Illegal Wildlife Tra,Thailand,15.8700,100.9925,Digital Governance and Innovation,SDG 11,Capacity Building,DNP- Min. of Natural Resources; Global Environ...,5183103.0,Finalisation,Completed,2018-05-01,2025-06-01,2018,2025,Sub-national / Local,Sector code 12.8; Sector code 15.2; Sector cod...,The Project Objective is to reduce the traffic...,Thailand_projects.xml
10,XM-DAC-41114-PROJECT-01002974,Accelerating COVID-19 recovery for vulnerable ...,Cambodia,12.5657,104.9910,Climate Resilience and Environment,SDG 1 / SDG 8; SDG 11; SDG 13; SDG 5,Capacity Building,"MINISTRY OF COMMERCE, CHINA; UNDP",0.0,Finalisation,Completed,2024-07-01,2025-11-01,2024,2025,Sub-national / Local,Sector code 1.1; Sector code 10.2; Sector code...,The COVID-19 pandemic brought substantive impa...,Cambodia_projects.xml
0,XM-DAC-41114-OUTPUT-00091274,A

In [49]:
print("Rows:", dashboard_sample.shape[0])
print("Columns:", dashboard_sample.shape[1])
print("Countries:", dashboard_sample["Country"].nunique())
print("Total Budget USD:", dashboard_sample["Budget USD"].sum())

print("\nCountries included:")
print(dashboard_sample["Country"].value_counts())

print("\nProject Status:")
print(dashboard_sample["Project Status"].value_counts())

print("\nThematic Focus:")
print(dashboard_sample["Thematic Focus"].value_counts())

print("\nSDG Alignment:")
print(dashboard_sample["SDG Alignment"].value_counts().head(10))

Rows: 100
Columns: 20
Countries: 10
Total Budget USD: 276204576.0

Countries included:
Country
Bangladesh          10
Cambodia            10
Indonesia           10
Nepal               10
Pakistan            10
Papua New Guinea    10
Philippines         10
Sri Lanka           10
Thailand            10
Viet Nam            10
Name: count, dtype: int64

Project Status:
Project Status
Finalisation    100
Name: count, dtype: int64

Thematic Focus:
Thematic Focus
Climate Resilience and Environment           39
Digital Governance and Innovation            16
Other Development Area                       16
Local Governance and Institutions            15
Gender Equality and Social Inclusion          6
Inclusive Economic Development                6
Social Services and Community Development     2
Name: count, dtype: int64

SDG Alignment:
SDG Alignment
Other SDG                26
SDG 13                   20
SDG 16                   10
SDG 11; SDG 16            5
SDG 5                     5
SDG 11 

In [50]:
print("Dashboard Status:")
print(dashboard_sample["Dashboard Status"].value_counts())

print("\nPreview:")
dashboard_sample[[
    "Project Title",
    "Country",
    "Project Status",
    "Dashboard Status",
    "Start Year",
    "End Year"
]].head(10)

Dashboard Status:
Dashboard Status
Completed    85
Closing      15
Name: count, dtype: int64

Preview:


,Project Title,Country,Project Status,Dashboard Status,Start Year,End Year
0,Activating Village Court II,Bangladesh,Finalisation,Completed,2016,2024
1,Activating Village Court Phase II,Bangladesh,Finalisation,Completed,2016,2024
2,Activating Village Court inCHT,Bangladesh,Finalisation,Completed,2019,2024
3,Community Cohesion in CXB,Bangladesh,Finalisation,Completed,2018,2024
4,DRRF-Disaster Resp& Recov.Faci,Bangladesh,Finalisation,Completed,2018,2024
5,Disaster Response and Recovery Facility DRRF,Bangladesh,Finalisation,Completed,2018,2025
6,Disaster Risk Managemnt in CXB,Bangladesh,Finalisation,Completed,2018,2024
7,Education and Skills in CHT,Bangladesh,Finalisation,Completed,2019,2024
8,Efficient and Accountable LG,Bangladesh,Finalisation,Completed,2017,2024
9,Efficient and Accountable Local Governance EALG,Bangladesh,Finalisation,Completed,2017,2024


## 8. Summary of Steps Completed in this Notebook

This notebook completed the data preparation workflow for the Asia Pacific Local Governance Project Mapping Dashboard. The main purpose was to transform public UNDP project data into a clean, structured, dashboard-ready Excel dataset for Power BI.

### 8.1 Library Import and Folder Setup

The notebook started by importing the required Python libraries for data handling, XML parsing, file management, text cleaning, date processing, and web data download. The project folder structure was then created using `pathlib`, including separate folders for raw data and processed output files.

This step helped organize the project professionally and made it easier to separate original source files from cleaned dashboard-ready data.

### 8.2 Public UNDP XML Data Source Selection

A list of selected Asia Pacific countries was defined, including Bangladesh, Cambodia, Indonesia, Nepal, Pakistan, Papua New Guinea, Philippines, Sri Lanka, Thailand, and Viet Nam.

For each country, a public UNDP Transparency Portal XML source link was added. These XML files provided project-level development data suitable for preparing a sample governance and local development project mapping dataset.

### 8.3 XML File Download

The notebook downloaded the public UNDP XML files for the selected countries and saved them in the raw data folder. A download function was used to loop through all country links, retrieve the files, handle possible download errors, and save each country file with a clear file name.

This step created the raw data foundation for the project.

### 8.4 XML Parsing and Field Extraction

The notebook parsed the downloaded XML files and extracted key project-level fields, including:

1. Project ID  
2. Project Title  
3. Country  
4. Project Status Code  
5. Start Date  
6. End Date  
7. Sector Information  
8. Donor or Funding Source  
9. Budget USD  
10. Project Description  
11. Source File  

Helper functions were created to safely extract text from XML elements, clean narrative text, and handle missing values.

### 8.5 Data Cleaning and Standardization

After extracting the raw project records, the notebook cleaned the dataset by standardizing text fields, converting date columns into proper date format, extracting start and end years, converting budget values into numeric format, and removing duplicate project records.

This step made the dataset more reliable and easier to use in Power BI.

### 8.6 Project Status Preparation

The original IATI project status code was converted into readable project status labels, such as Finalisation, Implementation, Closed, Cancelled, or Suspended.

Since the selected public sample mostly contained projects marked as Finalisation, an additional dashboard-friendly field named `Dashboard Status` was created using project start and end years. This field provides more useful status categories for Power BI filtering, including Completed and Closing.

The original source status was kept separately to maintain transparency.

### 8.7 Country Coordinates for Map Visualization

Country-level latitude and longitude values were added for each selected country. These coordinates will support geographic map visualization in Power BI.

This enables the dashboard to show project distribution across Asia Pacific countries using map visuals.

### 8.8 Analytical Classification Fields

Several dashboard-ready analytical fields were created using keyword-based classification from project titles, sector information, and project descriptions.

The generated fields include:

1. `SDG Alignment`  
2. `Thematic Focus`  
3. `Intervention Type`  
4. `Geographic Coverage`  

These fields help convert raw project descriptions into meaningful dashboard categories, allowing users to explore projects by SDG, theme, intervention type, and geographic scope.

### 8.9 Final Dashboard Dataset Creation

A final dashboard-ready table was created with 20 columns. The dataset was sorted and structured for use in Power BI.

The final sample included:

1. 100 project records  
2. 10 Asia Pacific countries  
3. 20 dashboard-ready fields  
4. Total budget of 276,204,576 USD  
5. Dashboard status categories: Completed and Closing  

### 8.10 Excel Export for Power BI

The cleaned dataset was exported as an Excel file named:

`asia_pacific_undp_project_mapping_sample.xlsx`

This file will be used as the main source file for the Power BI dashboard.

A separate metadata file was also created to document the source, purpose, countries included, date prepared, and methodology note.

### 8.11 Learning Outcome

This notebook demonstrated how Python can be used for professional data preparation before dashboard development. The workflow covered public data collection, XML parsing, data cleaning, text standardization, date processing, budget preparation, analytical classification, geographic tagging, and Excel export.

The cleaned Excel output is now ready to be imported into Power BI for building an interactive local governance project mapping dashboard.